# Analysis Notebook — Four Scenario Comparison\n\nReads the saved CSV files from B1–B4. This notebook does **not** run diffusion or SSA, so charts can be regenerated quickly after experiment runs.

In [ ]:
from pathlib import Path\nimport pandas as pd, numpy as np\nimport matplotlib.pyplot as plt\nROOT=Path.cwd()\nif ROOT.name=='notebooks': ROOT=ROOT.parent\nRESULTS=ROOT/'results'\nscenario_ids=['B1','B2','B3','B4']\nframes=[]\nfor sid in scenario_ids:\n    p=RESULTS/sid/'experiment_results.csv'\n    if p.exists():\n        d=pd.read_csv(p); d['scenario']=sid; frames.append(d)\nif not frames: raise FileNotFoundError('No scenario CSV files found yet.')\ndata=pd.concat(frames,ignore_index=True)\ndata.head()\n

In [ ]:
summary=(data.groupby(['scenario','K','variant','mode'],as_index=False)\n    .agg(net_advantage=('net_advantage','mean'),spread_A=('spread_A','mean'),spread_B=('spread_B','mean'),\n         blocked_B=('blocked_B','mean'),runtime=('ssa_runtime_sec','mean'),swap_runtime=('swap_runtime_sec','mean')))\nsummary.to_csv(RESULTS/'combined'/'combined_summary.csv',index=False)\nprint('Summary rows:',len(summary))\nsummary.head()\n

## 1. Net Advantage by scenario and objective

In [ ]:
for sid in scenario_ids:\n    for mode in sorted(data['mode'].dropna().unique()):\n        sub=summary[(summary.scenario==sid)&(summary.mode==mode)]\n        if sub.empty: continue\n        plt.figure(figsize=(11,6))\n        for variant in sub.variant.unique():\n            v=sub[sub.variant==variant].sort_values('K')\n            plt.plot(v.K,v.net_advantage,marker='o',label=variant)\n        plt.xlabel('K (seed budget)'); plt.ylabel('Net Advantage = Spread A − Spread B')\n        plt.title(f'{sid}: Net Advantage by K — {mode}'); plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.show()\n

## 2. Swap refinement change

In [ ]:
wide=data.pivot_table(index=['scenario','K','variant'],columns='mode',values='net_advantage').reset_index()\nif {'SSA only','SSA + Swap refinement'}.issubset(wide.columns):\n    wide['swap_delta']=wide['SSA + Swap refinement']-wide['SSA only']\n    display(wide.sort_values(['scenario','variant','K']))\n    wide.to_csv(RESULTS/'combined'/'swap_delta.csv',index=False)\n    for sid in scenario_ids:\n        sub=wide[wide.scenario==sid]\n        if sub.empty: continue\n        plt.figure(figsize=(10,5))\n        for variant in sub.variant.unique():\n            v=sub[sub.variant==variant].sort_values('K')\n            plt.plot(v.K,v.swap_delta,marker='o',label=variant)\n        plt.axhline(0,linewidth=1); plt.xlabel('K'); plt.ylabel('Swap Net Advantage improvement')\n        plt.title(f'{sid}: SSA + Swap minus SSA only'); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()\n

## 3. Scenario comparison for each objective

In [ ]:
for variant in data.variant.unique():\n    for mode in data.mode.unique():\n        sub=summary[(summary.variant==variant)& (summary.mode==mode)]\n        if sub.empty: continue\n        plt.figure(figsize=(10,5))\n        for sid in scenario_ids:\n            v=sub[sub.scenario==sid].sort_values('K')\n            if not v.empty: plt.plot(v.K,v.net_advantage,marker='o',label=sid)\n        plt.xlabel('K'); plt.ylabel('Net Advantage'); plt.title(f'{variant} — {mode} — scenario comparison')\n        plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()\n

## 4. Runtime overview

In [ ]:
runtime_summary=data.groupby(['scenario','K','variant','mode'],as_index=False).agg(\n    ssa_runtime=('ssa_runtime_sec','mean'),swap_runtime=('swap_runtime_sec','mean'),final_eval_runtime=('final_evaluation_runtime_sec','mean'))\ndisplay(runtime_summary.sort_values('ssa_runtime',ascending=False).head(20))\nruntime_summary.to_csv(RESULTS/'combined'/'runtime_summary.csv',index=False)\n